# NBT modelling summary

This notebook compares the winning data treatment, feature configuration and algorithm across the three prespecified prediction targets. Model selection occurred in development cross-validation; the values below are from the frozen untouched test sets.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULT_DIR = PROJECT_ROOT / "result" / "modeling"
winner_summary = pd.read_csv(RESULT_DIR / "cross_target_winner_summary.csv")
winner_summary

,target,winning missing strategy,winning feature configuration,winning model,primary metric,primary value,secondary metric,secondary value
0,duration_error_mins,"Missing-aware, priority retained",Both procedure levels,XGBoost,MAE,31.109098,R2,0.423620
1,operation_length_mins,"Missing-aware, priority retained",Both procedure levels,XGBoost,MAE,30.622747,R2,0.712434
2,meaningful_overrun_flag,"Missing-aware, priority retained",Both procedure levels,XGBoost,PR-AUC,0.797195,Recall,0.777656


## Winning type of data

The winner table states whether priority was retained through the selected missing strategy, which feature representation won, and whether the neural network or another algorithm performed best. Start hour, flagged-record exclusion and complete cases remain sensitivity analyses rather than primary data choices.

In [2]:
importance = pd.read_csv(RESULT_DIR / "cross_target_feature_importance.csv")
top_features = (
    importance.sort_values(["target", "rank"])
    .groupby("target", as_index=False, group_keys=False)
    .head(10)
)
top_features

,target,feature,importance mean,importance SD,rank
0,duration_error_mins,ExpectedDurationMins,14.374907,0.337394,1
1,duration_error_mins,anaesthetic_desc,4.490724,0.303627,2
2,duration_error_mins,procedure_code_group,3.993650,0.192238,3
3,duration_error_mins,procedure_code_category,3.586788,0.216280,4
4,duration_error_mins,intended_management_label,2.811918,0.192002,5
5,duration_error_mins,admission_type_label,2.303260,0.213652,6
6,duration_error_mins,ASAScore,0.560912,0.101060,7
7,duration_error_mins,age_at_operation,0.336198,0.089973,8
8,duration_error_mins,sex_national_code,0.222014,0.058485,9
9,duration_error_mins,priority_level_label,-0.008594,0.080863,10


## Interpretation

Permutation importance measures predictive contribution on the frozen test population and is not a causal effect. Correlated features can share importance. A low-ranked feature may still be clinically important, and external validation remains necessary before deployment.